# Metric — LLM-as-a-Judge Personalisation Success Rate (PSR)

$$\mathrm{PSR} = \frac{1}{N}\sum_{i} s_i, \qquad s_i = J\left(y^{+}_i, \hat{y}_i\right) \in \{0, 1\}$$

An external judge decides, per example, whether the generated answer satisfies
the **same implicit user preference** as the reference answer.

`JUDGE_MODEL = "deepseek-v4-pro"`, `JUDGE_TEMPERATURE = 0.0`,
`JUDGE_N_RUNS = 3` with per-run seeds; the reported label is the
**majority vote** over the three runs.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    get_ipython().run_line_magic("pip", "-q install -U pandas openai tqdm")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"

ADAPTER_EPOCH = 3

OUTPUT_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_llm_judge_pref_deepseek" / f"epoch_{ADAPTER_EPOCH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME, "| adapter epoch:", ADAPTER_EPOCH)
print("Output dir:", OUTPUT_DIR.resolve())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
Mounted at /content/drive
Subset: all_subset | adapter epoch: 3
Output dir: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_3


In [6]:
SUBSET_DIR = PP_ROOT / SUBSET_NAME
SOURCE_CONFIGS = {
    "zero_shot": {
        "results_dir": SUBSET_DIR / "v2_personamem_zero_shot_50_snippet",
    },

}

ACTIVE_SOURCE = None

SELECT_PERSONAS: list | None = None
SPLITS = ["val"]

JUDGE_MODEL = "deepseek-v4-pro"
JUDGE_BATCH_SIZE = 4

JUDGE_MAX_WORKERS = 400
JUDGE_MAX_RETRIES = 6

JUDGE_TEMPERATURE = 0.0
JUDGE_SEED = 42
DISABLE_THINKING = True

JUDGE_N_RUNS = 3
JUDGE_VARY_SEED_PER_RUN = True

print("Judge provider: deepseek | model:", JUDGE_MODEL, "| seed:", JUDGE_SEED, "| temp:", JUDGE_TEMPERATURE, "| n_runs:", JUDGE_N_RUNS)
print("Active source:", ACTIVE_SOURCE or f"all ({len(SOURCE_CONFIGS)})")
print("Configured sources:")
for k, cfg in SOURCE_CONFIGS.items():
    print(f"  {k}: {cfg['results_dir']}")

Judge provider: deepseek | model: deepseek-v4-pro | seed: 42 | temp: 0.0 | n_runs: 3
Active source: all (1)
Configured sources:
  zero_shot: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_zero_shot_50_snippet


In [9]:
import ast
import json
import pandas as pd

def load_predictions(results_dir, split):
    """Read the combined all-rows file (all_<split>_predictions.csv) for this epoch."""

    csv_path = results_dir / f"all_{split}_predictions.csv"
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    if "persona_id" not in df.columns:
        raise ValueError(f"{csv_path} is missing a persona_id column")
    df["persona_id"] = df["persona_id"].astype(str)
    df["split"] = split
    return df

def filter_personas(df, persona_ids):
    if persona_ids is None:
        return df
    keep = set(persona_ids)
    return df[df["persona_id"].astype(str).isin(keep)].reset_index(drop=True)

sources_to_run = (
    {ACTIVE_SOURCE: SOURCE_CONFIGS[ACTIVE_SOURCE]}
    if ACTIVE_SOURCE
    else SOURCE_CONFIGS
)

loaded: dict[str, dict[str, pd.DataFrame]] = {}
for name, cfg in sources_to_run.items():
    base_rd = cfg["results_dir"]
    rd = base_rd / f"epoch_{ADAPTER_EPOCH}" if ADAPTER_EPOCH is not None else base_rd
    rd = base_rd
    if not rd.exists():
        print(f"[skip] {name}: {rd} not found")
        continue
    persona_filter = [str(p) for p in SELECT_PERSONAS] if SELECT_PERSONAS else None
    loaded[name] = {}
    for split in SPLITS:
        df = load_predictions(rd, split)
        if df is None:
            print(f"  [skip] {name} {split}: no {split}_predictions.csv in {rd}")
            continue
        loaded[name][split] = filter_personas(df, persona_filter)
        print(f"  {name} {split}: {len(loaded[name][split])} rows")
    if not loaded[name]:
        del loaded[name]

if not loaded:
    raise RuntimeError("No source data loaded.")

  zero_shot val: 714 rows


In [12]:
import math
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from tqdm.auto import tqdm

JUDGE_PROMPT = (
"""
You are an expert evaluator of personalised assistant responses.

Your task is to determine whether the model-generated response is personalised
according to the user's implicit preference.

You are given:

1. A preference type describing the dimension of personalisation.
2. A correct answer that demonstrates satisfaction of the user's preference.
3. An incorrect answer that demonstrates failure to satisfy the user's preference.
4. A model-generated answer that must be evaluated.

Use the correct and incorrect answers as positive and negative reference examples.
Do not require the generated answer to match the wording, facts, structure, or
length of the correct answer exactly. Instead, infer the implicit user preference
illustrated by the correct answer and contrast it with the failure illustrated
by the incorrect answer.

Evaluation procedure:

1. Identify the implicit preference represented by the correct answer.
2. Identify how the incorrect answer fails to satisfy that preference.
3. Determine whether the generated answer follows the same preference as the
   correct answer and avoids the failure demonstrated by the incorrect answer.
4. Judge only the personalisation dimension represented by the preference type.
   Do not penalise harmless differences in wording or additional relevant details.
5. The generated answer should receive 1 only when there is clear evidence that
   it is adapted to the user's preference. A generally good but non-personalised
   answer should receive 0.
6. If the generated answer is ambiguous, contradictory, or does not provide
   enough evidence of the intended personalisation, return 0.

Preference type:
{pref_type}

Positive reference — satisfies the preference:
{correct_answer}

Negative reference — does not satisfy the preference:
{incorrect_answer}

Model-generated answer:
{generated_answer}

Does the model-generated answer satisfy the same implicit user preference as
the positive reference while avoiding the failure shown by the negative reference?

Return exactly one token:
1 = yes
0 = no
"""
)

def build_judge_messages(row):
    pref_type = str(row.get("preference", "unknown"))
    correct = str(row.get("correct_answer", ""))
    generated = str(row.get("generated_answer", ""))
    incorrect_answer = str(ast.literal_eval(row.get("incorrect_answers", ""))[-1])
    user_content = JUDGE_PROMPT.format(
        pref_type=pref_type,
        correct_answer=correct,
        generated_answer=generated,
        incorrect_answer=incorrect_answer
    )
    return [{"role": "user", "content": user_content}]

def _softmax_pair(logprob_0, logprob_1, raw_token):
    if logprob_0 is not None and logprob_1 is not None:
        m = max(logprob_0, logprob_1)
        e0 = math.exp(logprob_0 - m)
        e1 = math.exp(logprob_1 - m)
        total = e0 + e1
        return e0 / total, e1 / total
    if logprob_1 is not None:
        p1 = math.exp(logprob_1)
        return 1.0 - p1, p1
    if logprob_0 is not None:
        p0 = math.exp(logprob_0)
        return p0, 1.0 - p0
    if raw_token == "1":
        return 0.0, 1.0
    if raw_token == "0":
        return 1.0, 0.0
    return 0.5, 0.5

class PreferenceJudge:
    def __init__(self, model_name="deepseek-v4-pro"):
        self.model_name = model_name
        api_key = os.environ["DEEPSEEK_API_KEY"]
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

    def _error_result(self, row, seed, err):
        """Placeholder row when the API call fails after retries (or returns empty)."""
        out = dict(row)
        out["judge_token"] = None
        out["judge_prob_0"] = float("nan")
        out["judge_prob_1"] = float("nan")
        out["judge_satisfies_pref"] = float("nan")
        out["judge_seed"] = seed
        out["judge_system_fingerprint"] = None
        out["judge_api_error"] = 1
        out["judge_api_error_msg"] = str(err)[:500]
        return out

    def _score_one(self, row, seed=None):
        use_seed = JUDGE_SEED if seed is None else seed
        try:
            messages = build_judge_messages(row)
        except Exception as e:
            return self._error_result(row, use_seed, e)

        create_kwargs = dict(
            model=self.model_name,
            messages=messages,
            max_tokens=1,
            temperature=JUDGE_TEMPERATURE,
            seed=use_seed,
            logprobs=True,
            top_logprobs=20,
        )
        if DISABLE_THINKING:
            create_kwargs["extra_body"] = {"thinking": {"type": "disabled"}}
        resp = None
        last_err = None
        for attempt in range(JUDGE_MAX_RETRIES):
            try:
                resp = self.client.chat.completions.create(**create_kwargs)
                break
            except Exception as e:
                last_err = e
                create_kwargs.pop("extra_body", None)
                time.sleep(min(2 ** attempt, 30))
        if resp is None:
            return self._error_result(row, use_seed, last_err or "API returned no response")

        if not getattr(resp, "choices", None):
            return self._error_result(row, use_seed, "API response has no choices")
        choice = resp.choices[0]
        raw_token = (choice.message.content or "").strip()
        has_logprobs = bool(choice.logprobs and choice.logprobs.content)
        if not raw_token and not has_logprobs:
            return self._error_result(row, use_seed, "API response empty (no content/logprobs)")

        logprob_0 = None
        logprob_1 = None
        if has_logprobs:
            item = choice.logprobs.content[0]
            logprob_map = {item.token.strip(): item.logprob}
            if item.top_logprobs:
                for tl in item.top_logprobs:
                    logprob_map[tl.token.strip()] = tl.logprob
            logprob_0 = logprob_map.get("0")
            logprob_1 = logprob_map.get("1")

        p0, p1 = _softmax_pair(logprob_0, logprob_1, raw_token)
        token = "1" if p1 >= p0 else "0"
        out = dict(row)
        out["judge_token"] = token
        out["judge_prob_0"] = p0
        out["judge_prob_1"] = p1
        out["judge_satisfies_pref"] = int(token == "1")
        out["judge_seed"] = use_seed
        out["judge_system_fingerprint"] = getattr(resp, "system_fingerprint", None)
        out["judge_api_error"] = 0
        out["judge_api_error_msg"] = ""
        return out

    def judge_batch(self, rows, seed=None):
        return [self._score_one(row, seed=seed) for row in rows]

    def judge_dataframe(self, df, desc, seed=None):
        rows = [r.to_dict() for _, r in df.iterrows()]
        if not rows:
            return pd.DataFrame(rows)
        use_seed = JUDGE_SEED if seed is None else seed
        records: list[dict | None] = [None] * len(rows)
        max_workers = max(1, min(JUDGE_MAX_WORKERS, len(rows)))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_idx = {
                executor.submit(self._score_one, row, seed): i
                for i, row in enumerate(rows)
            }
            for future in tqdm(as_completed(future_to_idx), total=len(rows), desc=desc):
                idx = future_to_idx[future]
                try:
                    records[idx] = future.result()
                except Exception as e:
                    records[idx] = self._error_result(rows[idx], use_seed, e)
        return pd.DataFrame(records)
judge = PreferenceJudge(JUDGE_MODEL)
print(f"Using DeepSeek judge: {JUDGE_MODEL} | seed={JUDGE_SEED} | temp={JUDGE_TEMPERATURE} | "
      f"thinking={'off' if DISABLE_THINKING else 'on'}")

Using DeepSeek judge: deepseek-v4-pro | seed=42 | temp=0.0 | thinking=off


In [14]:
import numpy as np

RUN_IDS = list(range(1, JUDGE_N_RUNS + 1))

def judge_dataframe_multi(df, source_name, split):
    """Judge every row JUDGE_N_RUNS full passes.

    Each pass k produces per-row: judge_token_run{k}, judge_prob_1_run{k},
    judge_prob_0_run{k}, judge_satisfies_pref_run{k}, judge_api_error_run{k},
    and its own run-level satisfaction_rate_{k} = mean over *successful* rows.
    A per-row majority vote across the runs gives judge_satisfies_pref_majority.
    Returns (per_row_df, [satisfaction_rate_1, ...], [error_rate_1, ...]).
    """
    base = df.reset_index(drop=True).copy()
    run_rates: list[float] = []
    error_rates: list[float] = []
    for k in RUN_IDS:
        seed = (JUDGE_SEED + k - 1) if JUDGE_VARY_SEED_PER_RUN else JUDGE_SEED
        run_df = judge.judge_dataframe(df, desc=f"{source_name}/{split} run{k}", seed=seed)
        base[f"judge_token_run{k}"] = run_df["judge_token"].values
        base[f"judge_prob_1_run{k}"] = run_df["judge_prob_1"].values
        base[f"judge_prob_0_run{k}"] = run_df["judge_prob_0"].values
        base[f"judge_satisfies_pref_run{k}"] = run_df["judge_satisfies_pref"].values
        base[f"judge_api_error_run{k}"] = run_df["judge_api_error"].astype(int).values
        base[f"judge_api_error_msg_run{k}"] = run_df["judge_api_error_msg"].values
        base[f"judge_seed_run{k}"] = seed

        n = len(run_df)
        n_err = int(base[f"judge_api_error_run{k}"].sum())
        err_pct = 100.0 * n_err / n if n else 0.0
        error_rates.append(err_pct)
        ok = base[f"judge_api_error_run{k}"] == 0
        rate_k = float(base.loc[ok, f"judge_satisfies_pref_run{k}"].mean()) if ok.any() else float("nan")
        run_rates.append(rate_k)
        print(
            f"  {source_name} {split} run{k}: satisfaction_rate={rate_k:.4f} | "
            f"api_error={n_err}/{n} ({err_pct:.2f}%) (seed={seed})"
        )

    sat_cols = [f"judge_satisfies_pref_run{k}" for k in RUN_IDS]
    votes = base[sat_cols].sum(axis=1, min_count=1)
    n_valid = base[sat_cols].notna().sum(axis=1)
    base["judge_satisfies_pref_majority"] = (votes >= (n_valid / 2)).astype("Int64")
    base.loc[n_valid == 0, "judge_satisfies_pref_majority"] = pd.NA
    base["judge_prob_1_mean"] = base[[f"judge_prob_1_run{k}" for k in RUN_IDS]].mean(axis=1)
    base["judge_prob_0_mean"] = base[[f"judge_prob_0_run{k}" for k in RUN_IDS]].mean(axis=1)
    base["judge_satisfies_pref"] = base["judge_satisfies_pref_majority"]
    base["judge_prob_1"] = base["judge_prob_1_mean"]
    base["judge_prob_0"] = base["judge_prob_0_mean"]
    return base, run_rates, error_rates

judged: dict[str, dict[str, pd.DataFrame]] = {}
run_summary_rows: list[dict] = []

for source_name, splits in loaded.items():
    judged[source_name] = {}
    for split in SPLITS:
        df = splits[split]
        if df.empty:
            continue
        if "generated_answer" not in df.columns:
            raise ValueError(f"{source_name}/{split}: missing generated_answer column")

        out, run_rates, error_rates = judge_dataframe_multi(df, source_name, split)
        out_path = OUTPUT_DIR / f"{source_name}_{split}_judge_run2.csv"
        judged[source_name][split] = out

        majority_rate = float(pd.to_numeric(out["judge_satisfies_pref_majority"], errors="coerce").mean())
        rate_mean = float(np.nanmean(run_rates))
        rate_std = float(np.nanstd(run_rates, ddof=1)) if len(run_rates) > 1 else 0.0

        row = {"subset": SUBSET_NAME, "source": source_name, "split": split, "n": len(out)}
        for k, r, e in zip(RUN_IDS, run_rates, error_rates):
            row[f"satisfaction_rate_{k}"] = r
            row[f"api_error_pct_{k}"] = e
        row["majority_vote_satisfaction_rate"] = majority_rate
        row["satisfaction_rate_mean"] = rate_mean
        row["satisfaction_rate_std"] = rate_std
        row["api_error_pct_mean"] = float(np.mean(error_rates)) if error_rates else 0.0
        run_summary_rows.append(row)

        rates_str = ", ".join(f"{r:.4f}" for r in run_rates)
        errs_str = ", ".join(f"{e:.2f}%" for e in error_rates)
        print(f"{source_name} {split}: run rates=[{rates_str}] | api_error_pct=[{errs_str}] | "
              f"majority={majority_rate:.4f} | mean={rate_mean:.4f} +/- {rate_std:.4f}")
        print(f"  saved {out_path}")

run_summary = pd.DataFrame(run_summary_rows)
run_summary_path = OUTPUT_DIR / "judge_run_satisfaction_summary.csv"
print("\nSaved per-(subset,model) run summary ->", run_summary_path)
print(run_summary.round(4).to_string(index=False))

zero_shot/val run1:   0%|          | 0/714 [00:00<?, ?it/s]

  zero_shot val run1: satisfaction_rate=0.0938 | api_error=0/714 (0.00%) (seed=42)


zero_shot/val run2:   0%|          | 0/714 [00:00<?, ?it/s]

  zero_shot val run2: satisfaction_rate=0.0938 | api_error=0/714 (0.00%) (seed=43)


zero_shot/val run3:   0%|          | 0/714 [00:00<?, ?it/s]

  zero_shot val run3: satisfaction_rate=0.0910 | api_error=0/714 (0.00%) (seed=44)
zero_shot val: run rates=[0.0938, 0.0938, 0.0910] | api_error_pct=[0.00%, 0.00%, 0.00%] | majority=0.0938 | mean=0.0929 +/- 0.0016
  saved /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_3/zero_shot_val_judge_run2.csv

Saved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_3/judge_run_satisfaction_summary.csv
    subset    source split   n  satisfaction_rate_1  api_error_pct_1  satisfaction_rate_2  api_error_pct_2  satisfaction_rate_3  api_error_pct_3  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std  api_error_pct_mean
all_subset zero_shot   val 714               0.0938              0.0               0.0938              0.0                0.091              0.0                           0.0938                  0.0929                 0.

In [ ]:
aved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_3/judge_run_satisfaction_summary.csv
    subset                              source split   n  satisfaction_rate_1  api_error_pct_1  satisfaction_rate_2  api_error_pct_2  satisfaction_rate_3  api_error_pct_3  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std  api_error_pct_mean
all_subset                     centralized_sft   val 714               0.4137           0.1401               0.4118              0.0               0.4132              0.0                           0.4132                  0.4129                 0.0010              0.0467
all_subset                    centralized_orpo   val 714               0.4076           0.0000               0.4006              0.0               0.4062              0.0                           0.4076                  0.4048                 0.0037              0.0000
all_subset       centralized_ctx_contrast_orpo   val 714               0.4230           0.0000               0.4146              0.0               0.4188              0.0                           0.4160                  0.4188                 0.0042              0.0000
all_subset           centralized_combined_orpo   val 714               0.4412           0.0000               0.4314              0.0               0.4342              0.0                           0.4370                  0.4356                 0.0050              0.0000
all_subset        centralized_gen_edit_snippet   val 714               0.4692           0.0000               0.4622              0.0               0.4664              0.0                           0.4692                  0.4659                 0.0035              0.0000
all_subset               federated_sft_gradavg   val 714               0.3838           0.0000               0.3880              0.0               0.3852              0.0                           0.3838                  0.3856                 0.0021              0.0000
all_subset              federated_orpo_gradavg   val 714               0.3515           0.0000               0.3487              0.0               0.3389              0.0                           0.3487                  0.3464                 0.0066              0.0000
all_subset federated_ctx_contrast_orpo_gradavg   val 714               0.3599           0.0000               0.3599              0.0               0.3585              0.0                           0.3599                  0.3595                 0.0008              0.0000
all_subset     federated_combined_orpo_gradavg   val 714               0.3445           0.0000               0.3431              0.0               0.3445              0.0                           0.3431                  0.3441                 0.0008              0.0000
all_subset  federated_gen_edit_gradavg_snippet   val 714               0.4076           0.0000               0.4090              0.0               0.4090              0.0                           0.4090                  0.4085                 0.0008              0.0000

In [ ]:
Saved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_3/judge_run_satisfaction_summary.csv
    subset                              source split   n  satisfaction_rate_1  satisfaction_rate_2  satisfaction_rate_3  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
all_subset                     centralized_sft   val 714               0.4104               0.4132               0.4146                           0.4132                  0.4127                 0.0021
all_subset                    centralized_orpo   val 714               0.4034               0.4006               0.4062                           0.4062                  0.4034                 0.0028
all_subset       centralized_ctx_contrast_orpo   val 714               0.4132               0.4202               0.4146                           0.4160                  0.4160                 0.0037
all_subset           centralized_combined_orpo   val 714               0.4342               0.4356               0.4468                           0.4342                  0.4388                 0.0069
all_subset        centralized_gen_edit_snippet   val 714               0.4678               0.4678               0.4664                           0.4692                  0.4673                 0.0008
all_subset               federated_sft_gradavg   val 714               0.3824               0.3852               0.3852                           0.3838                  0.3842                 0.0016
all_subset              federated_orpo_gradavg   val 714               0.3473               0.3487               0.3515                           0.3473                  0.3492                 0.0021
all_subset federated_ctx_contrast_orpo_gradavg   val 714               0.3641               0.3585               0.3515                           0.3599                  0.3581                 0.0063
all_subset     federated_combined_orpo_gradavg   val 714               0.3473               0.3333               0.3445                           0.3445                  0.3417                 0.0074
all_subset  federated_gen_edit_gradavg_snippet   val 714               0.4076               0.4104               0.4076                           0.4090                  0.4085                 0.0016

In [ ]:
Saved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_3/judge_run_satisfaction_summary.csv
    subset                       source split   n  satisfaction_rate_1  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
all_subset centralized_gen_edit_snippet   val 714               0.5154                           0.5154                  0.5154                    0.0

In [ ]:
import ast

a = "['asma', 'bubu']"
my_list = ast.literal_eval(a)
print(my_list)
print(type(my_list))

In [ ]:
aved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/epoch_4/judge_run_satisfaction_summary.csv
    subset                        source split   n  satisfaction_rate_1  satisfaction_rate_2  satisfaction_rate_3  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
all_subset              centralized_orpo   val 714               0.4132               0.4090               0.4090                           0.4104                  0.4104                 0.0024
all_subset centralized_ctx_contrast_orpo   val 714               0.3866               0.3866               0.3838                           0.3866                  0.3856                 0.0016
all_subset     centralized_combined_orpo   val 714               0.4510               0.4510               0.4496                           0.4510                  0.4505                 0.0008

In [ ]:
2 epoch of simpo after 3 epoch sft
2 epoch of simpo after 4 epoch of sft

In [ ]:
!ls

drive  sample_data


In [ ]:
Saved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_llm_judge_pref_deepseek/judge_run_satisfaction_summary.csv
    subset                        source split   n  satisfaction_rate_1  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
all_subset               centralized_sft   val 714               0.4118                           0.4118                  0.4118                    0.0
all_subset              centralized_orpo   val 714               0.4118                           0.4118                  0.4118                    0.0
all_subset centralized_ctx_contrast_orpo   val 714               0.4398                           0.4398                  0.4398                    0.0
all_subset     centralized_combined_orpo   val 714               0.4244                           0.4244                  0.4244                    0.0

In [ ]:
aved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/subset_1/v2_personamem_llm_judge_pref_deepseek/judge_run_satisfaction_summary.csv
  subset                              source split   n  satisfaction_rate_1  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
subset_1                     centralized_sft   val 240               0.5708                           0.5708                  0.5708                    0.0
subset_1                    centralized_orpo   val 240               0.5333                           0.5333                  0.5333                    0.0
subset_1           centralized_combined_orpo   val 240               0.5333                           0.5333                  0.5333                    0.0
subset_1               federated_sft_gradavg   val 240               0.5500                           0.5500                  0.5500                    0.0
subset_1              federated_orpo_gradavg   val 240               0.5250                           0.5250                  0.5250                    0.0
subset_1 federated_ctx_contrast_orpo_gradavg   val 240               0.5542                           0.5542                  0.5542                    0.0
subset_1     federated_combined_orpo_gradavg   val 240               0.5458                           0.5458                  0.5458                    0.0

In [ ]:
aved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref_deepseek/judge_run_satisfaction_summary.csv
  subset                              source split   n  satisfaction_rate_1  satisfaction_rate_2  satisfaction_rate_3  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
subset_0                     centralized_sft   val 230               0.5043               0.5087               0.5000                           0.5000                  0.5043                 0.0043
subset_0                    centralized_orpo   val 230               0.5609               0.5609               0.5565                           0.5609                  0.5594                 0.0025
subset_0       centralized_ctx_contrast_orpo   val 230               0.5478               0.5478               0.5478                           0.5478                  0.5478                 0.0000
subset_0   centralized_ctx_contrast_neg_orpo   val 230               0.5435               0.5391               0.5435                           0.5435                  0.5420                 0.0025
subset_0               federated_sft_gradavg   val 230               0.4478               0.4522               0.4522                           0.4522                  0.4507                 0.0025
subset_0              federated_orpo_gradavg   val 230               0.5174               0.5174               0.5130                           0.5174                  0.5159                 0.0025
subset_0 federated_ctx_contrast_orpo_gradavg   val 230               0.4217               0.4261               0.4217                           0.4261                  0.4232                 0.0025

Saved per-(subset,model) run summary -> /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref_deepseek/judge_run_satisfaction_summary.csv
  subset                          source split   n  satisfaction_rate_1  satisfaction_rate_2  satisfaction_rate_3  majority_vote_satisfaction_rate  satisfaction_rate_mean  satisfaction_rate_std
subset_0       centralized_combined_orpo   val 230               0.5957               0.6000               0.5957                           0.6000                  0.5971                 0.0025
subset_0 federated_combined_orpo_gradavg   val 230               0.5609               0.5609               0.5739                           0.5696                  0.5652                 0.0075

In [ ]:
summary_rows = []
for source_name, splits in judged.items():
    for split, df in splits.items():
        summary_rows.append({
            "subset": SUBSET_NAME,
            "source": source_name,
            "split": split,
            "persona_id": "ALL",
            "n": len(df),
            "judge_satisfaction_rate": df["judge_satisfies_pref"].mean(),
            "mean_judge_prob_1": df["judge_prob_1"].mean(),
            "mean_judge_prob_0": df["judge_prob_0"].mean(),
        })
        for pid, g in df.groupby("persona_id"):
            summary_rows.append({
                "subset": SUBSET_NAME,
                "source": source_name,
                "split": split,
                "persona_id": pid,
                "n": len(g),
                "judge_satisfaction_rate": g["judge_satisfies_pref"].mean(),
                "mean_judge_prob_1": g["judge_prob_1"].mean(),
                "mean_judge_prob_0": g["judge_prob_0"].mean(),
            })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "judge_summary.csv", index=False)

overall = summary[summary["persona_id"] == "ALL"].drop(columns=["persona_id"]).reset_index(drop=True)
overall.to_csv(OUTPUT_DIR / "judge_overall_summary.csv", index=False)

per_persona = summary[summary["persona_id"] != "ALL"].reset_index(drop=True)
per_persona.to_csv(OUTPUT_DIR / "judge_per_persona_summary.csv", index=False)

print("Saved overall ->", OUTPUT_DIR / "judge_overall_summary.csv")
print("Saved per-persona ->", OUTPUT_DIR / "judge_per_persona_summary.csv")
print("\n=== Overall (per model) satisfaction on", SUBSET_NAME, "===")
print(overall[["source", "split", "n", "judge_satisfaction_rate"]].round(4).to_string(index=False))

with open(OUTPUT_DIR / "judge_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "judge_provider": "deepseek",
        "judge_model": JUDGE_MODEL,
        "judge_seed": JUDGE_SEED,
        "judge_temperature": JUDGE_TEMPERATURE,
        "disable_thinking": DISABLE_THINKING,
        "judge_n_runs": JUDGE_N_RUNS,
        "vary_seed_per_run": JUDGE_VARY_SEED_PER_RUN,
        "subset": SUBSET_NAME,
        "sources": list(judged.keys()),
        "source_dirs": {k: str(SOURCE_CONFIGS[k]["results_dir"]) for k in judged.keys()},
        "select_personas": SELECT_PERSONAS,
    }, f, indent=2)

Saved overall -> /content/drive/MyDrive/privacy_perserving_pllm/subset_1/v2_personamem_llm_judge_pref_deepseek/judge_overall_summary.csv
Saved per-persona -> /content/drive/MyDrive/privacy_perserving_pllm/subset_1/v2_personamem_llm_judge_pref_deepseek/judge_per_persona_summary.csv

=== Overall (per model) satisfaction on subset_1 ===
                             source split   n  judge_satisfaction_rate
                    centralized_sft   val 240                   0.5083
                   centralized_orpo   val 240                   0.4625
      centralized_ctx_contrast_orpo   val 240                   0.4917
          centralized_combined_orpo   val 240                   0.4500
              federated_sft_gradavg   val 240                   0.5000
             federated_orpo_gradavg   val 240                   0.4292
federated_ctx_contrast_orpo_gradavg   val 240                   0.5000
    federated_combined_orpo_gradavg   val 240                   0.4833


In [ ]:
for source_name, splits in judged.items():
    for split, df in splits.items():
        if "pref_type" not in df.columns:
            continue
        by_pref = df.groupby("pref_type").agg(
            n=("judge_satisfies_pref", "count"),
            satisfaction_rate=("judge_satisfies_pref", "mean"),
            mean_prob_1=("judge_prob_1", "mean"),
        ).sort_values("n", ascending=False)
        print(f"\n=== {source_name} / {split} by pref_type ===")
        print(by_pref.head(15).round(4).to_string())
        by_pref.to_csv(OUTPUT_DIR / f"{source_name}_{split}_judge_by_pref_type.csv")


=== centralized_sft / val by pref_type ===
                                n  satisfaction_rate  mean_prob_1
pref_type                                                        
anti_stereotypical_pref        44             0.4545       0.4697
ask_to_forget                  43             0.4419       0.4419
neutral_preferences            37             0.5135       0.5135
health_and_medical_conditions  33             0.4242       0.4040
sensitive_info                 33             0.8485       0.8586
therapy_background             30             0.5333       0.5222
stereotypical_pref             20             0.3000       0.3000

=== centralized_orpo / val by pref_type ===
                                n  satisfaction_rate  mean_prob_1
pref_type                                                        
anti_stereotypical_pref        44             0.5000       0.5076
ask_to_forget                  43             0.3488       0.3333
neutral_preferences            37             0.3784 

In [ ]:
_SFT_KEY = "centralized_sft"
_ORPO_KEY = "centralized_orpo"
_OVERLAP_SPLIT = "val"

def _overlap_key_cols(df):
    candidates = ["persona_id", "user_query", "correct_answer", "preference", "incorrect_answer"]
    return [c for c in candidates if c in df.columns]

if (
    _SFT_KEY in judged and _ORPO_KEY in judged
    and _OVERLAP_SPLIT in judged[_SFT_KEY] and _OVERLAP_SPLIT in judged[_ORPO_KEY]
):
    sft_df = judged[_SFT_KEY][_OVERLAP_SPLIT].copy()
    orpo_df = judged[_ORPO_KEY][_OVERLAP_SPLIT].copy()

    key_cols = _overlap_key_cols(sft_df)
    sft_df["_occ"] = sft_df.groupby(key_cols).cumcount()
    orpo_df["_occ"] = orpo_df.groupby(key_cols).cumcount()
    merge_keys = key_cols + ["_occ"]

    merged = sft_df[merge_keys + ["judge_satisfies_pref"]].merge(
        orpo_df[merge_keys + ["judge_satisfies_pref"]],
        on=merge_keys, how="inner", suffixes=("_sft", "_orpo"),
    )
    n_matched = len(merged)
    print(f"Matched {n_matched} datapoints (sft={len(sft_df)}, orpo={len(orpo_df)}) on keys={key_cols}")

    s = merged["judge_satisfies_pref_sft"].astype(int)
    o = merged["judge_satisfies_pref_orpo"].astype(int)
    both_1 = int(((s == 1) & (o == 1)).sum())
    both_0 = int(((s == 0) & (o == 0)).sum())
    sft1_orpo0 = int(((s == 1) & (o == 0)).sum())
    sft0_orpo1 = int(((s == 0) & (o == 1)).sum())
    agree = both_1 + both_0

    print(f"\n=== ORPO vs SFT satisfaction overlap ({_OVERLAP_SPLIT}, epoch {ADAPTER_EPOCH}) ===")
    print(f"  both satisfied   (1/1): {both_1}")
    print(f"  both unsatisfied (0/0): {both_0}")
    print(f"  SFT=1, ORPO=0        : {sft1_orpo0}")
    print(f"  SFT=0, ORPO=1        : {sft0_orpo1}")
    if n_matched:
        print(f"  agree (overlap)      : {agree}/{n_matched} ({agree / n_matched:.1%})")
    print(f"  disagree             : {sft1_orpo0 + sft0_orpo1}/{n_matched}")

    ct = pd.crosstab(s, o, rownames=["SFT"], colnames=["ORPO"], margins=True)
    print("\nContingency table (rows=SFT, cols=ORPO):")
    print(ct.to_string())

    overlap_summary = pd.DataFrame([{
        "subset": SUBSET_NAME,
        "adapter_epoch": ADAPTER_EPOCH,
        "split": _OVERLAP_SPLIT,
        "n_matched": n_matched,
        "both_satisfied_1_1": both_1,
        "both_unsatisfied_0_0": both_0,
        "sft1_orpo0": sft1_orpo0,
        "sft0_orpo1": sft0_orpo1,
        "agreement_rate": (agree / n_matched) if n_matched else float("nan"),
    }])
    overlap_path = OUTPUT_DIR / f"orpo_vs_sft_overlap_{_OVERLAP_SPLIT}.csv"
    overlap_summary.to_csv(overlap_path, index=False)
    merged.to_csv(OUTPUT_DIR / f"orpo_vs_sft_rowlevel_{_OVERLAP_SPLIT}.csv", index=False)
    print("\nSaved overlap summary ->", overlap_path)
else:
    print(f"[skip] need both '{_SFT_KEY}' and '{_ORPO_KEY}' judged for split '{_OVERLAP_SPLIT}'. "
          f"Have: {list(judged)}")